In [ ]:
import pandas as pd
import numpy as np
import logging

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')


In [ ]:
# Replace with your file paths or upload in Colab
call_logs = pd.read_csv('call_logs.csv')
agent_roster = pd.read_csv('agent_roster.csv')
disposition_summary = pd.read_csv('disposition_summary.csv')

logging.info("CSV files loaded successfully.")


In [ ]:
required_columns = ['call_date', 'agent_id', 'org_id']

for col in required_columns:
    for df_name, df in [('call_logs', call_logs), ('disposition_summary', disposition_summary)]:
        if col not in df.columns:
            logging.error(f"{col} missing in {df_name}")
        elif df[col].isnull().any():
            logging.warning(f"Missing values in {col} of {df_name}")
        elif df.duplicated(subset=[col]).any():
            logging.warning(f"Duplicate {col} entries in {df_name}")

# Convert to datetime
call_logs['call_date'] = pd.to_datetime(call_logs['call_date'], errors='coerce')
disposition_summary['call_date'] = pd.to_datetime(disposition_summary['call_date'], errors='coerce')


In [ ]:
# Merge call_logs + disposition_summary on agent_id, org_id, call_date
merged = call_logs.merge(disposition_summary, on=['agent_id', 'org_id', 'call_date'], how='left')

# Merge with agent_roster on agent_id, org_id
merged = merged.merge(agent_roster, on=['agent_id', 'org_id'], how='left')

logging.info(f"Merged dataset shape: {merged.shape}")


In [ ]:
# Add helper columns
merged['call_completed'] = merged['status'].apply(lambda x: 1 if x.lower() == 'completed' else 0)
merged['presence'] = merged['login_time'].notnull().astype(int)

# Group and aggregate
summary = merged.groupby(['call_date', 'agent_id', 'users_first_name', 'users_last_name']).agg(
    total_calls=('call_id', 'count'),
    unique_loans_contacted=('installment_id', pd.Series.nunique),
    completed_calls=('call_completed', 'sum'),
    avg_duration_min=('duration', lambda x: round(x.mean() / 60, 2)),
    presence=('presence', 'max')
).reset_index()

summary['connect_rate'] = round((summary['completed_calls'] / summary['total_calls']) * 100, 2)

logging.info("Feature engineering completed.")
summary.head()


,call_date,agent_id,users_first_name,users_last_name,total_calls,unique_loans_contacted,completed_calls,avg_duration_min,presence,connect_rate
0,2025-04-28,A001,AgentFirst1,AgentLast1,20,20,2,0.11,1,10.00
1,2025-04-28,A002,AgentFirst2,AgentLast2,23,23,3,0.13,1,13.04
2,2025-04-28,A003,AgentFirst3,AgentLast3,21,21,8,0.12,1,38.10
3,2025-04-28,A004,AgentFirst4,AgentLast4,27,27,4,0.13,1,14.81
4,2025-04-28,A005,AgentFirst5,AgentLast5,29,28,4,0.12,0,13.79


In [ ]:
summary.to_csv('agent_performance_summary.csv', index=False)
logging.info("Summary CSV saved as 'agent_performance_summary.csv'.")

# Slack-style summary message
report_date = summary['call_date'].max().strftime('%Y-%m-%d')
top_agent = summary.loc[summary['connect_rate'].idxmax()]
avg_duration = round(summary['avg_duration_min'].mean(), 1)
total_agents = summary['agent_id'].nunique()

print(f"Agent Summary for {report_date}\n"
      f"Top Performer: {top_agent['users_first_name']} {top_agent['users_last_name']} ({top_agent['connect_rate']}% connect rate)\n"
      f"Total Active Agents: {total_agents}\n"
      f"Average Duration: {avg_duration} min")


Agent Summary for 2025-04-28
Top Performer: AgentFirst3 AgentLast3 (38.1% connect rate)
Total Active Agents: 20
Average Duration: 0.1 min
